<a href="https://colab.research.google.com/github/allefbcc/Pesquisa-TCC/blob/main/masterPipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline de Previsão de Despesas

Pipeline completo de modelagem preditiva:
- **Etapa 1** — Redes Neurais: LSTM isolado + TFT
- **Etapa 2** — Modelos Clássicos: Regressão Linear, SVR, GBRT, LWR, KNN, Nearest(1-NN), AutoARIMA
- **Etapa 3** — Visualizações comparativas

In [17]:
!pip install optuna

In [18]:
!pip install pmdarima

In [19]:
!pip install darts

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.7/65.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.0/396.0 kB 12.8 MB/s eta 0:00:00


In [20]:
import os
import random
import warnings
import logging
from typing import Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing

import optuna

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

try:
    from darts import TimeSeries
    from darts.models import TFTModel
    from pytorch_lightning.callbacks import EarlyStopping as PLEarlyStopping
    HAS_DARTS = True
except ImportError:
    HAS_DARTS = False
    print("[WARN] darts não instalado — TFT será ignorado. Instale com: pip install darts")

try:
    import pmdarima as pm
    HAS_PMDARIMA = True
except ImportError:
    HAS_PMDARIMA = False
    print("[WARN] pmdarima não instalado — AutoARIMA será ignorado. Instale com: pip install pmdarima")

try:
    from statsmodels.tsa.seasonal import STL
    HAS_STL = True
except ImportError:
    HAS_STL = False
    print("[WARN] statsmodels não disponível — decomposição STL será omitida")

[WARN] darts não instalado — TFT será ignorado. Instale com: pip install darts


In [21]:
# Supressão de logs
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
tf.get_logger().setLevel("ERROR")
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("darts").setLevel(logging.ERROR)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Caminhos e splits temporais
DATA_PATH = "https://raw.githubusercontent.com/allefbcc/Pesquisa-TCC/refs/heads/main/S%C3%A9rie%20Temporal/dbp_2025_correction.csv"
TRAIN_END = "2015-12"
VAL_END   = "2016-12"
TEST_END  = "2019-12"

# Hiperparâmetros globais
SEASONAL_LAG    = 12
WINDOW_SIZE     = 12
N_TRIALS        = 50       # modelos clássicos — mais trials pois rodam rápido
N_TRIALS_LSTM   = 25
N_TRIALS_TFT    = 12       # TFT é mais caro de treinar
EPOCHS_MAX_LSTM = 200
EPOCHS_MAX_TFT  = 80
BATCH_SIZE      = 16

# Diretórios de saída
OUTPUT_DIR = "./outputs"
CHARTS_DIR = "./outputs/charts"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHARTS_DIR, exist_ok=True)

# Caminhos dos CSVs intermediários (usados na Etapa 3)
CLASSICOS_CSV = "./outputs/classicos_previsoes.csv"
NEURAIS_CSV   = "./outputs/neurais_previsoes.csv"

# Mapa de cores para visualizações
COLORS = {
    "Real":            "black",
    "LSTM isolado":    "#d62728",
    "TFT":             "#6a3d9a",
    "Media Movel":     "green",
    "AutoARIMA":       "#1f77b4",
    "Linear Regression":     "#2ca02c",
    "SVR":             "#ff7f0e",
    "GBRT":            "#9467bd",
    "Lazy (LWR)":      "#8c564b",
    "Nearest (1-NN)":  "#e377c2",
    "KNN":             "#17becf",
    "Suav. Exp. HW":   "#bcbd22",
}

NEURAL_MODELS  = ["LSTM isolado", "TFT"]
CLASSIC_MODELS = ["AutoARIMA", "Linear Regression", "SVR", "GBRT", "Lazy (LWR)", "Nearest (1-NN)", "KNN",  "Media Movel", "Suav. Exp. HW",]

## Funções Utilitárias

Pré-processamento e métricas compartilhados por todas as etapas do pipeline.

In [22]:
def load_data(path):
    df = pd.read_csv(path)
    df.columns = ["date", "value"]
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").set_index("date").asfreq("MS")
    return df


def split_temporal(df):
    train   = df.loc[:TRAIN_END].copy()
    val     = df.loc[pd.to_datetime(TRAIN_END) + pd.offsets.MonthBegin(1):VAL_END].copy()
    test    = df.loc[pd.to_datetime(VAL_END)   + pd.offsets.MonthBegin(1):TEST_END].copy()
    holdout = df.loc[pd.to_datetime(TEST_END)  + pd.offsets.MonthBegin(1):].copy()
    return train, val, test, holdout


def seasonal_diff(y, lag=SEASONAL_LAG):
    z = np.full_like(y, np.nan, dtype=float)
    z[lag:] = y[lag:] - y[:-lag]
    return z


def fit_scaler_on_train(z_full, n_train):
    z_tr = z_full[SEASONAL_LAG:n_train]
    z_tr = z_tr[~np.isnan(z_tr)].reshape(-1, 1)
    sc = MinMaxScaler(feature_range=(-1, 1)).fit(z_tr)
    return sc


def make_supervised(z_scaled, window=WINDOW_SIZE):
    Xs, ys, idxs = [], [], []
    for i in range(window, len(z_scaled)):
        win = z_scaled[i - window:i]
        tgt = z_scaled[i]
        if np.isnan(win).any() or np.isnan(tgt):
            continue
        Xs.append(win); ys.append(tgt); idxs.append(i)
    return np.array(Xs), np.array(ys), np.array(idxs)


def metrics(y_true, y_pred) -> Dict[str, float]:
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    valid = ~(np.isnan(yt) | np.isnan(yp))
    yt, yp = yt[valid], yp[valid]
    if len(yt) == 0:
        return {"MAE": np.nan, "RMSE": np.nan, "MAPE (%)": np.nan, "Bias (%)": np.nan}
    return {
        "MAE":      mean_absolute_error(yt, yp),
        "RMSE":     float(np.sqrt(mean_squared_error(yt, yp))),
        "MAPE (%)": float(np.mean(np.abs((yt - yp) / yt)) * 100),
        "Bias (%)": float(np.mean((yt - yp) / yt) * 100),
    }

## Etapa 1 — Redes Neurais (LSTM + TFT)

Treina e avalia dois modelos neurais de forma independente:
- **LSTM isolado**: Keras/TF, otimizado via Optuna no conjunto de validação.
- **TFT** (Temporal Fusion Transformer): via darts, modo univariado.

Avaliação:
- **Test (2017–2019)**: walk-forward 1-step-ahead
- **Holdout (>2019)**: forecast recursivo multi-step

Saídas: `neurais_metrics.csv`, `neurais_previsoes.csv`, `neurais_comparativo.png`

In [23]:
def to_seq(X):
    return X.reshape(X.shape[0], X.shape[1], 1)


# =================================================================== #
# LSTM (Keras)
# =================================================================== #
def build_lstm(units, dropout, lr, window=WINDOW_SIZE):
    model = Sequential([
        Input(shape=(window, 1)),
        LSTM(units, return_sequences=False),
        Dropout(dropout),
        Dense(16, activation="relu"),
        Dense(1),
    ])
    model.compile(optimizer=Adam(learning_rate=lr), loss="mse")
    return model


def fit_lstm(model, Xt, yt, Xv=None, yv=None, epochs=EPOCHS_MAX_LSTM,
             batch_size=BATCH_SIZE, patience=15):
    val_data = (Xv, yv) if Xv is not None and len(Xv) else None
    monitor = "val_loss" if val_data else "loss"
    cb = [EarlyStopping(monitor=monitor, patience=patience,
                        restore_best_weights=True)]
    model.fit(Xt, yt, validation_data=val_data, epochs=epochs,
              batch_size=batch_size, callbacks=cb, verbose=0, shuffle=False)
    return model


def objective_lstm(trial, Xt, yt, Xv, yv):
    units   = trial.suggest_int("units", 16, 128, step=16)
    dropout = trial.suggest_float("dropout", 0.0, 0.4)
    lr      = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    bs      = trial.suggest_categorical("batch_size", [8, 16, 32])
    tf.keras.backend.clear_session()
    model = build_lstm(units, dropout, lr)
    model = fit_lstm(model, Xt, yt, Xv, yv, batch_size=bs, patience=12)
    pred = model.predict(Xv, verbose=0).ravel()
    return mean_squared_error(yv, pred)


def lstm_predict_one(model, window_scaled):
    x_seq = window_scaled.reshape(1, -1, 1)
    return float(model.predict(x_seq, verbose=0).ravel()[0])


def walk_forward_lstm(df_known, test_dates, model, scaler):
    y_all = df_known["value"].values.astype(float)
    dates_all = df_known.index
    z = seasonal_diff(y_all, SEASONAL_LAG)
    z_scaled = np.full_like(z, np.nan, dtype=float)
    mask = ~np.isnan(z)
    z_scaled[mask] = scaler.transform(z[mask].reshape(-1, 1)).ravel()
    preds = []
    for d in test_dates:
        t = dates_all.get_loc(d)
        win = z_scaled[t - WINDOW_SIZE:t]
        if np.isnan(win).any():
            preds.append(np.nan); continue
        z_hat_s = lstm_predict_one(model, win)
        z_hat   = scaler.inverse_transform([[z_hat_s]])[0][0]
        y_hat   = z_hat + y_all[t - SEASONAL_LAG]
        preds.append(y_hat)
    return pd.Series(preds, index=test_dates)


def recursive_lstm(df_known, horizon_dates, model, scaler):
    y_hist = list(df_known["value"].values.astype(float))
    z_hist = list(seasonal_diff(np.array(y_hist), SEASONAL_LAG))
    z_arr  = np.array(z_hist)
    z_scaled_hist = np.full_like(z_arr, np.nan, dtype=float)
    mask = ~np.isnan(z_arr)
    z_scaled_hist[mask] = scaler.transform(z_arr[mask].reshape(-1, 1)).ravel()
    z_scaled_hist = list(z_scaled_hist)
    preds = []
    for _ in horizon_dates:
        win = np.array(z_scaled_hist[-WINDOW_SIZE:])
        z_hat_s = lstm_predict_one(model, win)
        z_hat   = scaler.inverse_transform([[z_hat_s]])[0][0]
        y_hat   = z_hat + y_hist[-SEASONAL_LAG]
        preds.append(y_hat)
        y_hist.append(y_hat); z_hist.append(z_hat); z_scaled_hist.append(z_hat_s)
    return pd.Series(preds, index=horizon_dates)


# =================================================================== #
# TFT (darts)
# =================================================================== #
def build_z_timeseries(df_known, scaler):
    """TimeSeries do darts contendo z_scaled (sem NaN)."""
    y = df_known["value"].values.astype(float)
    z = seasonal_diff(y, SEASONAL_LAG)
    z_scaled = np.full_like(z, np.nan, dtype=float)
    mask = ~np.isnan(z)
    z_scaled[mask] = scaler.transform(z[mask].reshape(-1, 1)).ravel()
    valid = ~np.isnan(z_scaled)
    dates_valid = df_known.index[valid]
    values_valid = z_scaled[valid]
    return TimeSeries.from_times_and_values(dates_valid, values_valid, freq="MS")


def split_ts_by_dates(ts_full, tcut_ms, vcut_ms, test_end_ms):
    """
    Divide a TimeSeries do darts em train/val/tv/until_test usando split_after().
    A indexação por máscara booleana (ts[mask]) NÃO é suportada pelo darts atual.
    split_after(t) inclui o ponto t no primeiro pedaço.
    """
    ts_train, ts_after_train = ts_full.split_after(tcut_ms)
    ts_val,   _              = ts_after_train.split_after(vcut_ms)
    ts_tv,    _              = ts_full.split_after(vcut_ms)
    ts_until_test, _         = ts_full.split_after(test_end_ms)
    return ts_train, ts_val, ts_tv, ts_until_test


def build_tft(hidden_size, lstm_layers, num_heads, dropout, lr, n_epochs,
              patience=15):
    es = PLEarlyStopping(monitor="train_loss", patience=patience,
                         min_delta=1e-4, mode="min")
    return TFTModel(
        input_chunk_length=WINDOW_SIZE,
        output_chunk_length=1,
        hidden_size=hidden_size,
        lstm_layers=lstm_layers,
        num_attention_heads=num_heads,
        dropout=dropout,
        batch_size=BATCH_SIZE,
        n_epochs=n_epochs,
        optimizer_kwargs={"lr": lr},
        random_state=SEED,
        add_relative_index=True,        # única feature derivada (univariado)
        pl_trainer_kwargs={
            "accelerator": "cpu",
            "enable_progress_bar": False,
            "enable_model_summary": False,
            "callbacks": [es],
            "logger": False,
        },
    )


def objective_tft(trial, ts_train, ts_val):
    hidden_size  = trial.suggest_categorical("hidden_size", [8, 16, 32])
    lstm_layers  = trial.suggest_int("lstm_layers", 1, 2)
    num_heads    = trial.suggest_categorical("num_heads", [1, 2, 4])
    dropout      = trial.suggest_float("dropout", 0.0, 0.3)
    lr           = trial.suggest_float("lr", 1e-4, 1e-2, log=True)

    model = build_tft(hidden_size, lstm_layers, num_heads, dropout, lr,
                      n_epochs=EPOCHS_MAX_TFT, patience=12)
    try:
        model.fit(ts_train, verbose=False)
    except Exception:
        return float("inf")

    # Avaliação no val: predict recursivo de len(ts_val) passos
    pred = model.predict(n=len(ts_val))
    val_arr  = ts_val.values().ravel()
    pred_arr = pred.values().ravel()
    n = min(len(val_arr), len(pred_arr))
    return float(mean_squared_error(val_arr[:n], pred_arr[:n]))


def walk_forward_tft_test(model, ts_full, test_dates, scaler, df_known):
    """historical_forecasts: walk-forward 1-step-ahead nativo do darts."""
    bt = model.historical_forecasts(
        series=ts_full,
        start=pd.Timestamp(test_dates[0]),
        forecast_horizon=1, stride=1,
        retrain=False, last_points_only=True,
        verbose=False,
    )
    z_preds_s = bt.values().ravel()
    # Normaliza para DatetimeIndex compatível com test_dates
    bt_dates  = pd.to_datetime(pd.DatetimeIndex(bt.time_index)).tz_localize(None)

    y_all = df_known["value"].values.astype(float)
    dates_all = df_known.index
    out = {}
    for d, z_s in zip(bt_dates, z_preds_s):
        z_hat = scaler.inverse_transform([[z_s]])[0][0]
        try:
            t_idx = dates_all.get_loc(d)
        except KeyError:
            continue
        out[pd.Timestamp(d)] = z_hat + y_all[t_idx - SEASONAL_LAG]
    s = pd.Series(out)
    s.index = pd.to_datetime(s.index)
    return s.reindex(pd.to_datetime(test_dates))


def recursive_tft(model, ts_until_test, horizon_dates, scaler, df_known):
    """TFT prevê n passos; revertemos z->y respeitando o lag-12 recursivo."""
    n = len(horizon_dates)
    pred_ts = model.predict(n=n, series=ts_until_test)
    z_preds_s = pred_ts.values().ravel()

    y_hist = list(df_known["value"].values.astype(float))
    preds = []
    for i, z_s in enumerate(z_preds_s):
        z_hat = scaler.inverse_transform([[z_s]])[0][0]
        # y_{t-12} é real nos primeiros 12 meses do holdout, depois é previsão
        y_lag = y_hist[-SEASONAL_LAG + i] if i < SEASONAL_LAG else preds[i - SEASONAL_LAG]
        y_hat = z_hat + y_lag
        preds.append(y_hat)
    return pd.Series(preds, index=horizon_dates)


# =================================================================== #
# Main
# =================================================================== #
def main():
    print("[LOAD] base...")
    df = load_data(DATA_PATH)
    train, val, test, holdout = split_temporal(df)
    print(f"   train={len(train)} val={len(val)} test={len(test)} holdout={len(holdout)}")

    # ---------- PREPROCESSAMENTO ----------
    df_known = pd.concat([train, val, test])
    y_known  = df_known["value"].values.astype(float)
    z = seasonal_diff(y_known, SEASONAL_LAG)
    scaler = fit_scaler_on_train(z, n_train=len(train))
    z_scaled = np.full_like(z, np.nan, dtype=float)
    mask = ~np.isnan(z)
    z_scaled[mask] = scaler.transform(z[mask].reshape(-1, 1)).ravel()

    Xall, yall, idx_all = make_supervised(z_scaled, WINDOW_SIZE)
    target_dates = df_known.index[idx_all]
    tcut = pd.to_datetime(TRAIN_END) + pd.offsets.MonthEnd(0)
    vcut = pd.to_datetime(VAL_END)   + pd.offsets.MonthEnd(0)
    mask_tr  = target_dates <= tcut
    mask_val = (target_dates > tcut) & (target_dates <= vcut)
    X_tr,  y_tr  = Xall[mask_tr],  yall[mask_tr]
    X_val, y_val = Xall[mask_val], yall[mask_val]
    X_tr_seq, X_val_seq = to_seq(X_tr), to_seq(X_val)
    print(f"   X_train={X_tr.shape} X_val={X_val.shape}")

    rows = []
    all_preds = {}  # nome -> dict(test=..., holdout=...)

    # ==================== LSTM ISOLADO ====================
    print(f"\n[OPTUNA] LSTM isolado ({N_TRIALS_LSTM} trials)...")
    study = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(lambda t: objective_lstm(t, X_tr_seq, y_tr, X_val_seq, y_val),
                   n_trials=N_TRIALS_LSTM, show_progress_bar=False)
    p_lstm = study.best_params
    print(f"   best MSE val = {study.best_value:.6f} | params = {p_lstm}")

    print("[FIT] LSTM final (train+val, early stopping)...")
    tf.keras.backend.clear_session()
    lstm = build_lstm(p_lstm["units"], p_lstm["dropout"], p_lstm["lr"])
    lstm = fit_lstm(lstm, X_tr_seq, y_tr, X_val_seq, y_val,
                    batch_size=p_lstm["batch_size"], patience=20)

    print("[INFER] LSTM walk-forward (test) + recursivo (holdout)...")
    pt = walk_forward_lstm(df_known, test.index, lstm, scaler)
    ph = (recursive_lstm(df_known, holdout.index, lstm, scaler)
          if len(holdout) else pd.Series(dtype=float))
    all_preds["LSTM isolado"] = {"test": pt, "holdout": ph}
    mt = metrics(test["value"].values, pt.values)
    mh = (metrics(holdout["value"].values, ph.values) if len(holdout)
          else {k: np.nan for k in ["MAE","RMSE","MAPE (%)","Bias (%)"]})
    print(f"   TEST    MAE={mt['MAE']:.1f}  MAPE={mt['MAPE (%)']:.2f}%")
    print(f"   HOLDOUT MAE={mh['MAE']:.1f}  MAPE={mh['MAPE (%)']:.2f}%")
    rows.append({"Modelo": "LSTM isolado",
                 "Test MAE": mt["MAE"], "Test RMSE": mt["RMSE"],
                 "Test MAPE (%)": mt["MAPE (%)"],
                 "Hold MAE": mh["MAE"], "Hold RMSE": mh["RMSE"],
                 "Hold MAPE (%)": mh["MAPE (%)"],
                 "Hold Bias (%)": mh["Bias (%)"]})

    # ==================== TFT ====================
    if HAS_DARTS:
        print(f"\n[OPTUNA] TFT ({N_TRIALS_TFT} trials — esse passo é o mais lento)...")
        ts_full = build_z_timeseries(df_known, scaler)

        # IMPORTANTE: split_after() em vez de boolean indexing
        # (não suportado em darts >= 0.27)
        tcut_ms     = pd.to_datetime(TRAIN_END)     # 2015-12-01
        vcut_ms     = pd.to_datetime(VAL_END)       # 2016-12-01
        test_end_ms = pd.to_datetime(TEST_END)      # 2019-12-01
        ts_train, ts_val_d, ts_tv, ts_until_test = split_ts_by_dates(
            ts_full, tcut_ms, vcut_ms, test_end_ms
        )
        print(f"   ts_train={len(ts_train)}  ts_val={len(ts_val_d)}  "
              f"ts_tv={len(ts_tv)}  ts_until_test={len(ts_until_test)}")

        study_t = optuna.create_study(direction="minimize",
                                      sampler=optuna.samplers.TPESampler(seed=SEED))
        study_t.optimize(lambda t: objective_tft(t, ts_train, ts_val_d),
                         n_trials=N_TRIALS_TFT, show_progress_bar=False)
        p_tft = study_t.best_params
        print(f"   best MSE val = {study_t.best_value:.6f} | params = {p_tft}")

        print("[FIT] TFT final (train+val)...")
        tft = build_tft(p_tft["hidden_size"], p_tft["lstm_layers"],
                        p_tft["num_heads"], p_tft["dropout"], p_tft["lr"],
                        n_epochs=EPOCHS_MAX_TFT, patience=20)
        tft.fit(ts_tv, verbose=False)

        print("[INFER] TFT walk-forward (test)...")
        try:
            pt_t = walk_forward_tft_test(tft, ts_full, test.index, scaler, df_known)
        except Exception as e:
            print(f"   [ERR] walk_forward_tft_test falhou: {e}")
            pt_t = pd.Series([np.nan] * len(test.index), index=test.index)

        print("[INFER] TFT recursivo (holdout)...")
        if len(holdout):
            try:
                ph_t = recursive_tft(tft, ts_until_test, holdout.index,
                                     scaler, df_known)
            except Exception as e:
                print(f"   [ERR] recursive_tft falhou: {e}")
                ph_t = pd.Series([np.nan] * len(holdout.index), index=holdout.index)
        else:
            ph_t = pd.Series(dtype=float)

        all_preds["TFT"] = {"test": pt_t, "holdout": ph_t}
        mt = metrics(test["value"].values, pt_t.values)
        mh = (metrics(holdout["value"].values, ph_t.values) if len(holdout)
              else {k: np.nan for k in ["MAE","RMSE","MAPE (%)","Bias (%)"]})
        print(f"   TEST    MAE={mt['MAE']:.1f}  MAPE={mt['MAPE (%)']:.2f}%")
        print(f"   HOLDOUT MAE={mh['MAE']:.1f}  MAPE={mh['MAPE (%)']:.2f}%")
        rows.append({"Modelo": "TFT",
                     "Test MAE": mt["MAE"], "Test RMSE": mt["RMSE"],
                     "Test MAPE (%)": mt["MAPE (%)"],
                     "Hold MAE": mh["MAE"], "Hold RMSE": mh["RMSE"],
                     "Hold MAPE (%)": mh["MAPE (%)"],
                     "Hold Bias (%)": mh["Bias (%)"]})
    else:
        print("[SKIP] TFT (darts não instalado)")

    # ---------- SAÍDAS ----------
    summary = pd.DataFrame(rows).sort_values("Test MAPE (%)")
    summary_path = os.path.join(OUTPUT_DIR, "neurais_metrics.csv")
    summary.to_csv(summary_path, index=False)
    print("\n========================== RESUMO ==========================")
    print(summary.round(3).to_string(index=False))
    print(f"\n[OUT] {summary_path}")

    # CSV consolidado das previsões
    pred_cols = {"real": df["value"]}
    for name, d in all_preds.items():
        s = pd.concat([d["test"], d["holdout"]]).reindex(df.index)
        pred_cols[name] = s
    pred_df = pd.DataFrame(pred_cols)
    pred_csv = os.path.join(OUTPUT_DIR, "neurais_previsoes.csv")
    pred_df.to_csv(pred_csv)
    print(f"[OUT] {pred_csv}")

    # Plot rápido
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(df.index, df["value"], color="black", lw=1.3, label="Realizado")
    palette = {"LSTM isolado": "#d62728", "TFT": "#6a3d9a"}
    for name, d in all_preds.items():
        s = pd.concat([d["test"], d["holdout"]])
        ax.plot(s.index, s.values, color=palette.get(name, "gray"),
                lw=1.6, label=name, alpha=0.85)
    for v in [TRAIN_END, VAL_END, TEST_END]:
        ax.axvline(pd.to_datetime(v) + pd.offsets.MonthEnd(0),
                   color="gray", linestyle=":", alpha=0.6)
    ax.set_title("Redes Neurais — LSTM isolado vs TFT")
    ax.set_xlabel("Data"); ax.set_ylabel("Despesa")
    ax.legend(loc="upper left"); ax.grid(alpha=0.3)
    out_png = os.path.join(OUTPUT_DIR, "neurais_comparativo.png")
    plt.tight_layout(); plt.savefig(out_png, dpi=130); plt.close()
    print(f"[OUT] {out_png}")

    return summary, all_preds

main()

[LOAD] base...
   train=149 val=12 test=36 holdout=72
   X_train=(125, 12) X_val=(12, 12)

[OPTUNA] LSTM isolado (25 trials)...
   best MSE val = 0.242601 | params = {'units': 80, 'dropout': 0.06820964947491662, 'lr': 0.00013492834268013249, 'batch_size': 16}
[FIT] LSTM final (train+val, early stopping)...
[INFER] LSTM walk-forward (test) + recursivo (holdout)...
   TEST    MAE=2305.6  MAPE=4.87%
   HOLDOUT MAE=11520.9  MAPE=16.07%
[SKIP] TFT (darts não instalado)

========================== RESUMO ==========================
      Modelo  Test MAE  Test RMSE  Test MAPE (%)  Hold MAE  Hold RMSE  Hold MAPE (%)  Hold Bias (%)
LSTM isolado  2305.614   2642.067          4.872 11520.896   14222.03         16.067           -5.4

[OUT] ./outputs/neurais_metrics.csv
[OUT] ./outputs/neurais_previsoes.csv
[OUT] ./outputs/neurais_comparativo.png


(         Modelo     Test MAE    Test RMSE  Test MAPE (%)      Hold MAE  \
 0  LSTM isolado  2305.614094  2642.067484       4.871639  11520.895839   
 
       Hold RMSE  Hold MAPE (%)  Hold Bias (%)  
 0  14222.029902      16.066574      -5.400147  ,
 {'LSTM isolado': {'test': date
   2017-01-01    41595.258327
   2017-02-01    43901.184821
   2017-03-01    44540.756897
   2017-04-01    44376.303267
   2017-05-01    46107.617951
   2017-06-01    44713.040472
   2017-07-01    44862.104548
   2017-08-01    49275.733159
   2017-09-01    58275.645004
   2017-10-01    44970.329224
   2017-11-01    53058.959768
   2017-12-01    59051.988348
   2018-01-01    45789.553235
   2018-02-01    47351.651753
   2018-03-01    47501.995301
   2018-04-01    48602.833446
   2018-05-01    52836.596503
   2018-06-01    47908.606678
   2018-07-01    48441.095001
   2018-08-01    52484.630723
   2018-09-01    63780.931806
   2018-10-01    49320.490727
   2018-11-01    53575.265718
   2018-12-01    64141.4608

## Etapa 2 — Modelos Clássicos

Treina e avalia 7 modelos com otimização de hiperparâmetros via Optuna:
Regressão Linear, Média Móvel, Suavização Exponencial de Holt-Winters, SVR, GBRT, LWR (Locally Weighted Regression), KNN, Nearest(1-NN), Auto-ARIMA.

Saídas: `classicos_metrics.csv`, `classicos_previsoes.csv`, `classicos_comparativo.png`, `classicos_test_2017_2019.png`

In [24]:
# ================================================================= #
# SIMPLE MOVING AVERAGE
# ================================================================= #
class SimpleMovingAverage:

    def __init__(self, k=12):
        self.k = k

    def fit(self, X, y):
        return self

    def predict(self, X):

        X = np.atleast_2d(X)

        preds = []

        for row in X:
            preds.append(np.mean(row[-self.k:]))

        return np.array(preds)



# ================================================================= #
# 2. LAZY LEARNING — Locally Weighted Regression
# ================================================================= #
class LocallyWeightedRegression:
    """
    LWR clássico (Atkeson, 1997). Para cada query x*:
        - calcula pesos w_i = exp(-||x_i - x*||² / (2h²))
        - resolve regressão linear ponderada local
    Custo de O(N·d) por query — OK para a escala desta série.
    """
    def __init__(self, bandwidth: float = 1.0, ridge: float = 1e-6):
        self.h = bandwidth
        self.ridge = ridge

    def fit(self, X, y):
        self.X_ = np.asarray(X, dtype=float)
        self.y_ = np.asarray(y, dtype=float)
        return self

    def predict(self, X_query):
        X_query = np.atleast_2d(X_query).astype(float)
        N, d = self.X_.shape
        X_aug = np.hstack([np.ones((N, 1)), self.X_])
        preds = np.empty(len(X_query))
        I = np.eye(d + 1) * self.ridge
        for j, x in enumerate(X_query):
            d2 = np.sum((self.X_ - x) ** 2, axis=1)
            w  = np.exp(-d2 / (2.0 * self.h ** 2 + 1e-12))
            W  = w[:, None]                          # broadcasting
            try:
                A = (X_aug * W).T @ X_aug + I
                b = (X_aug * W).T @ self.y_
                theta = np.linalg.solve(A, b)
                x_aug = np.concatenate([[1.0], x])
                preds[j] = float(x_aug @ theta)
            except np.linalg.LinAlgError:
                preds[j] = float(np.sum(w * self.y_) / (np.sum(w) + 1e-12))
        return preds


# ================================================================= #
# 3. OPTUNA OBJECTIVES
# ================================================================= #

def objective_hw(trial, y_train, y_val):

    trend = trial.suggest_categorical("trend", ["add", "mul"])
    seasonal = trial.suggest_categorical("seasonal", ["add", "mul"])
    seasonal_periods = trial.suggest_categorical(
        "seasonal_periods",
        [6, 12, 24]
    )

    try:
        model = ExponentialSmoothing(
            y_train,
            trend=trend,
            seasonal=seasonal,
            seasonal_periods=seasonal_periods
        )

        fit = model.fit(optimized=True)

        pred = fit.forecast(len(y_val))

        return mean_squared_error(y_val, pred)

    except Exception:
        return float("inf")

def fit_hw(y_train_val, params):

    model = ExponentialSmoothing(
        y_train_val,
        trend=params["trend"],
        seasonal=params["seasonal"],
        seasonal_periods=params["seasonal_periods"]
    )

    return model.fit(optimized=True)

def walk_forward_hw(y_train_val, y_test, params):

    history = list(y_train_val)

    preds = []

    for actual in y_test:

        try:
            model = ExponentialSmoothing(
                history,
                trend=params["trend"],
                seasonal=params["seasonal"],
                seasonal_periods=params["seasonal_periods"]
            )

            fit = model.fit(optimized=True)

            fc = fit.forecast(1)

            pred = float(fc[0])

        except Exception:
            pred = np.nan

        preds.append(pred)

        # atualiza com valor REAL
        history.append(actual)

    return np.array(preds)

def recursive_hw(y_history, horizon, params):

    history = list(y_history)

    preds = []

    for _ in range(horizon):

        try:
            model = ExponentialSmoothing(
                history,
                trend=params["trend"],
                seasonal=params["seasonal"],
                seasonal_periods=params["seasonal_periods"]
            )

            fit = model.fit(optimized=True)

            fc = fit.forecast(1)

            pred = float(fc[0])

        except Exception:
            pred = np.nan

        preds.append(pred)

        # atualiza com previsão
        history.append(pred)

    return np.array(preds)

def run_optuna_hw(y_train, y_val):

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )

    study.optimize(
        lambda t: objective_hw(t, y_train, y_val),
        n_trials=N_TRIALS,
        show_progress_bar=False
    )

    print(
        f"   [OPTUNA] HW MSE val = {study.best_value:.6f} | "
        f"params = {study.best_params}"
    )

    return study.best_params


def objective_sma(trial, Xt, yt, Xv, yv):

    k = trial.suggest_int(
        "k",
        2,
        min(WINDOW_SIZE, 24)
    )

    model = SimpleMovingAverage(k=k)

    pred = model.predict(Xv)

    return _mse(yv, pred)


def _mse(y, p): return mean_squared_error(y, p)


def objective_linear(Xt, yt, Xv, yv):
    m = LinearRegression().fit(Xt, yt)
    return _mse(yv, m.predict(Xv))


def objective_svr(trial, Xt, yt, Xv, yv):
    C       = trial.suggest_float("C", 1e-2, 1e2, log=True)
    gamma   = trial.suggest_float("gamma", 1e-4, 1.0, log=True)
    epsilon = trial.suggest_float("epsilon", 1e-4, 5e-1, log=True)
    m = SVR(kernel="rbf", C=C, gamma=gamma, epsilon=epsilon).fit(Xt, yt)
    return _mse(yv, m.predict(Xv))


def objective_gbrt(trial, Xt, yt, Xv, yv):
    n_estimators     = trial.suggest_int("n_estimators", 100, 1000, step=50)
    max_depth        = trial.suggest_int("max_depth", 2, 6)
    lr               = trial.suggest_float("lr", 1e-3, 3e-1, log=True)
    subsample        = trial.suggest_float("subsample", 0.6, 1.0)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    m = GradientBoostingRegressor(
        n_estimators=n_estimators, max_depth=max_depth,
        learning_rate=lr, subsample=subsample,
        min_samples_leaf=min_samples_leaf, random_state=SEED).fit(Xt, yt)
    return _mse(yv, m.predict(Xv))


def objective_lwr(trial, Xt, yt, Xv, yv):
    bw = trial.suggest_float("bandwidth", 0.05, 5.0, log=True)
    m  = LocallyWeightedRegression(bandwidth=bw).fit(Xt, yt)
    return _mse(yv, m.predict(Xv))


def objective_knn(trial, Xt, yt, Xv, yv):
    k = trial.suggest_int("n_neighbors", 2, min(20, len(Xt) - 1))
    m = KNeighborsRegressor(n_neighbors=k, weights="uniform").fit(Xt, yt)
    return _mse(yv, m.predict(Xv))


def run_optuna(name, objective_fn, Xt, yt, Xv, yv, n_trials=N_TRIALS):
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    study.optimize(lambda t: objective_fn(t, Xt, yt, Xv, yv),
                   n_trials=n_trials, show_progress_bar=False)
    print(f"   [OPTUNA] {name:<6} MSE val = {study.best_value:.6f} | "
          f"params = {study.best_params}")
    return study.best_params


# ================================================================= #
# 4. WALK-FORWARD GENÉRICO (modelos tabulares)
# ================================================================= #
def predict_one(model, window_scaled) -> float:
    return float(model.predict(window_scaled.reshape(1, -1))[0])


def walk_forward_test(df_known, test_dates, model, scaler) -> pd.Series:
    """Walk-forward 1-step-ahead — janelas com z reais até t-1."""
    y_all = df_known["value"].values.astype(float)
    dates_all = df_known.index

    z = seasonal_diff(y_all, SEASONAL_LAG)
    z_scaled = np.full_like(z, np.nan, dtype=float)
    mask = ~np.isnan(z)
    z_scaled[mask] = scaler.transform(z[mask].reshape(-1, 1)).ravel()

    preds = []
    for d in test_dates:
        t = dates_all.get_loc(d)
        win = z_scaled[t - WINDOW_SIZE:t]
        if np.isnan(win).any():
            preds.append(np.nan); continue
        z_hat_s = predict_one(model, win)
        z_hat   = scaler.inverse_transform([[z_hat_s]])[0][0]
        y_hat   = z_hat + y_all[t - SEASONAL_LAG]
        preds.append(y_hat)
    return pd.Series(preds, index=test_dates)


def recursive_forecast_holdout(df_known, horizon_dates, model, scaler) -> pd.Series:
    """Recursive multi-step; previsões alimentam o histórico."""
    y_hist = list(df_known["value"].values.astype(float))
    z_hist = list(seasonal_diff(np.array(y_hist), SEASONAL_LAG))
    z_arr  = np.array(z_hist)
    z_scaled_hist = np.full_like(z_arr, np.nan, dtype=float)
    mask = ~np.isnan(z_arr)
    z_scaled_hist[mask] = scaler.transform(z_arr[mask].reshape(-1, 1)).ravel()
    z_scaled_hist = list(z_scaled_hist)

    preds = []
    for _ in horizon_dates:
        win = np.array(z_scaled_hist[-WINDOW_SIZE:])
        z_hat_s = predict_one(model, win)
        z_hat   = scaler.inverse_transform([[z_hat_s]])[0][0]
        y_hat   = z_hat + y_hist[-SEASONAL_LAG]
        preds.append(y_hat)
        y_hist.append(y_hat); z_hist.append(z_hat); z_scaled_hist.append(z_hat_s)
    return pd.Series(preds, index=horizon_dates)


# ================================================================= #
# 5. AUTOARIMA (pipeline independente)
# ================================================================= #
def fit_autoarima(y_train_val):
    print("   [AutoARIMA] auto-busca de (p,d,q)(P,D,Q,12) via stepwise...")
    model = pm.auto_arima(
        y_train_val,
        seasonal=True, m=SEASONAL_LAG,
        start_p=0, start_q=0, max_p=3, max_q=3,
        start_P=0, start_Q=0, max_P=2, max_Q=2,
        d=None, D=None, max_d=2, max_D=1,
        information_criterion="aicc",
        stepwise=True, suppress_warnings=True,
        error_action="ignore", trace=False, n_jobs=1,
    )
    print(f"   [AutoARIMA] selecionado: {model.order} x {model.seasonal_order}")
    return model


def walk_forward_arima(model, y_test) -> np.ndarray:
    """Para cada t, prevê 1 passo e em seguida atualiza com o valor real."""
    preds = []
    for actual in y_test:
        fc = model.predict(n_periods=1)
        preds.append(float(np.asarray(fc).ravel()[0]))
        model.update([actual])
    return np.array(preds)


def forecast_arima(model, n_periods) -> np.ndarray:
    return np.asarray(model.predict(n_periods=n_periods)).ravel()



# ================================================================= #
# 7. GRÁFICO TESTE (2017–2019) — REAL vs PREVISTOS
# ================================================================= #
def plot_test_predictions(df, test, all_preds, out_path):

    fig, ax = plt.subplots(figsize=(14, 6))

    # Série real apenas no TEST
    ax.plot(
        test.index,
        test["value"],
        color="black",
        lw=2.2,
        label="Real"
    )

    # Previsões dos modelos
    for name, d in all_preds.items():

        pt = d["test"]

        ax.plot(
            pt.index,
            pt.values,
            lw=1.7,
            alpha=0.85,
            color=COLORS.get(name, None),
            label=name
        )

    # Destaque visual do período
    ax.axvspan(
        test.index.min(),
        test.index.max(),
        color="green",
        alpha=0.04
    )

    ax.set_title(
        "Período de Teste (2017–2019) — Real vs Previsto",
        fontsize=14,
        fontweight="bold"
    )

    ax.set_xlabel("Data")
    ax.set_ylabel("Despesa")

    ax.grid(alpha=0.3)

    ax.legend(
        loc="upper left",
        fontsize=9,
        ncol=2
    )

    plt.tight_layout()
    plt.savefig(out_path, dpi=130)
    plt.close()

    print(f"[OUT] gráfico salvo em {out_path}")

# ================================================================= #
# 7. PLOT COMPARATIVO
# ================================================================= #
def plot_comparativo(df_full, all_preds: Dict[str, dict], out_path):
    models = list(all_preds.keys())
    n = len(models)
    ncols = 2
    nrows = (n + 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.2 * nrows), sharex=True)
    axes = np.atleast_2d(axes).ravel()

    for ax, name in zip(axes, models):
        ptest = all_preds[name]["test"]
        phold = all_preds[name]["holdout"]
        ax.plot(df_full.index, df_full["value"], color="black", lw=1.1, label="Real")
        if len(ptest):
            ax.plot(ptest.index, ptest.values, color="tab:blue", lw=1.4, label="Test (WF)")
        if len(phold):
            ax.plot(phold.index, phold.values, color="tab:red", lw=1.4,
                    linestyle="--", label="Holdout (recursivo)")
        for v in [TRAIN_END, VAL_END, TEST_END]:
            ax.axvline(pd.to_datetime(v) + pd.offsets.MonthEnd(0),
                       color="gray", linestyle=":", alpha=0.5)
        ax.set_title(name); ax.grid(alpha=0.3)
        ax.legend(fontsize=8, loc="upper left")
    for ax in axes[len(models):]:
        ax.axis("off")
    plt.suptitle("Modelos Clássicos — Realizado vs Previsto", fontsize=13)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(out_path, dpi=120); plt.close()
    print(f"\n[PLOT] {out_path}")



# ================================================================= #
# 8. MAIN
# ================================================================= #
def main():
    print("[LOAD] base...")
    df = load_data(DATA_PATH)
    train, val, test, holdout = split_temporal(df)
    print(f"   train={len(train)}  val={len(val)}  test={len(test)}  holdout={len(holdout)}")

    # ---------- PRÉ-PROCESSAMENTO COMUM ----------
    df_known = pd.concat([train, val, test])
    y_known  = df_known["value"].values.astype(float)
    z = seasonal_diff(y_known, SEASONAL_LAG)
    scaler = fit_scaler_on_train(z, n_train=len(train))

    z_scaled = np.full_like(z, np.nan, dtype=float)
    mask = ~np.isnan(z)
    z_scaled[mask] = scaler.transform(z[mask].reshape(-1, 1)).ravel()

    Xall, yall, idx_all = make_supervised(z_scaled, WINDOW_SIZE)
    target_dates = df_known.index[idx_all]
    train_cut = pd.to_datetime(TRAIN_END) + pd.offsets.MonthEnd(0)
    val_cut   = pd.to_datetime(VAL_END)   + pd.offsets.MonthEnd(0)
    mask_tr  = target_dates <= train_cut
    mask_val = (target_dates > train_cut) & (target_dates <= val_cut)
    X_tr,  y_tr  = Xall[mask_tr],  yall[mask_tr]
    X_val, y_val = Xall[mask_val], yall[mask_val]
    print(f"   X_train={X_tr.shape}  X_val={X_val.shape}")

    # Para o treino final, treinamos em train+val (mais dados, hparams fixos)
    X_tv = np.vstack([X_tr, X_val])
    y_tv = np.concatenate([y_tr, y_val])

    # ---------- OPTUNA por modelo ----------
    print("\n[OPTUNA] otimizando hparams (val = 2016)...")
    print("   [LinearRegression] sem hiperparâmetros para otimizar")
    p_svr   = run_optuna("SVR",   objective_svr,   X_tr, y_tr, X_val, y_val)
    p_gbrt  = run_optuna("GBRT",  objective_gbrt,  X_tr, y_tr, X_val, y_val)
    p_lwr   = run_optuna("LWR",   objective_lwr,   X_tr, y_tr, X_val, y_val)
    p_knn   = run_optuna("KNN",   objective_knn,   X_tr, y_tr, X_val, y_val)
    p_sma = run_optuna("SMA",     objective_sma,   X_tr, y_tr, X_val, y_val)
    p_hw = run_optuna_hw(train["value"].values, val["value"].values)


    # ---------- TREINO FINAL com train+val ----------
    print("\n[FIT] treinando modelos finais com train+val...")
    models = {
        "Linear Regression": LinearRegression().fit(X_tv, y_tv),
        "SVR":     SVR(kernel="rbf", C=p_svr["C"], gamma=p_svr["gamma"],
                       epsilon=p_svr["epsilon"]).fit(X_tv, y_tv),
        "GBRT":    GradientBoostingRegressor(
                       n_estimators=p_gbrt["n_estimators"],
                       max_depth=p_gbrt["max_depth"],
                       learning_rate=p_gbrt["lr"],
                       subsample=p_gbrt["subsample"],
                       min_samples_leaf=p_gbrt["min_samples_leaf"],
                       random_state=SEED).fit(X_tv, y_tv),
        "Lazy (LWR)":      LocallyWeightedRegression(bandwidth=p_lwr["bandwidth"]).fit(X_tv, y_tv),
        "Nearest (1-NN)":  KNeighborsRegressor(n_neighbors=1).fit(X_tv, y_tv),
        "KNN":     KNeighborsRegressor(n_neighbors=p_knn["n_neighbors"],
                                       weights="uniform").fit(X_tv, y_tv),
        "Media Movel": SimpleMovingAverage(k=p_sma["k"]),


    }

    # ---------- INFERÊNCIA: walk-forward (test) + recursivo (holdout) ----------
    all_preds = {}
    rows = []
    for name, model in models.items():
        print(f"\n[INFER] {name}")
        pt = walk_forward_test(df_known, test.index, model, scaler)
        ph = (recursive_forecast_holdout(df_known, holdout.index, model, scaler)
              if len(holdout) else pd.Series(dtype=float))
        all_preds[name] = {"test": pt, "holdout": ph}
        mt = metrics(test["value"].values, pt.values)
        mh = (metrics(holdout["value"].values, ph.values)
              if len(holdout) else {k: np.nan for k in ["MAE","RMSE","MAPE (%)","Bias (%)"]})
        print(f"   TEST    MAE={mt['MAE']:.1f}  MAPE={mt['MAPE (%)']:.2f}%")
        print(f"   HOLDOUT MAE={mh['MAE']:.1f}  MAPE={mh['MAPE (%)']:.2f}%")
        rows.append({"Modelo": name,
                     "Test MAE": mt["MAE"], "Test RMSE": mt["RMSE"],
                     "Test MAPE (%)": mt["MAPE (%)"],
                     "Hold MAE": mh["MAE"], "Hold RMSE": mh["RMSE"],
                     "Hold MAPE (%)": mh["MAPE (%)"],
                     "Hold Bias (%)": mh["Bias (%)"]})

    # ---------- AUTOARIMA (pipeline próprio) ----------
    if HAS_PMDARIMA:
        print("\n[INFER] AutoARIMA")
        y_train_val = pd.concat([train, val])["value"].values.astype(float)
        arima = fit_autoarima(y_train_val)

        pt_arr = walk_forward_arima(arima, test["value"].values.astype(float))
        pt_arima = pd.Series(pt_arr, index=test.index)

        if len(holdout):
            ph_arr = forecast_arima(arima, n_periods=len(holdout))
            ph_arima = pd.Series(ph_arr, index=holdout.index)
        else:
            ph_arima = pd.Series(dtype=float)

        all_preds["AutoARIMA"] = {"test": pt_arima, "holdout": ph_arima}
        mt = metrics(test["value"].values, pt_arima.values)
        mh = (metrics(holdout["value"].values, ph_arima.values)
              if len(holdout) else {k: np.nan for k in ["MAE","RMSE","MAPE (%)","Bias (%)"]})
        print(f"   TEST    MAE={mt['MAE']:.1f}  MAPE={mt['MAPE (%)']:.2f}%")
        print(f"   HOLDOUT MAE={mh['MAE']:.1f}  MAPE={mh['MAPE (%)']:.2f}%")
        rows.append({"Modelo": "AutoARIMA",
                     "Test MAE": mt["MAE"], "Test RMSE": mt["RMSE"],
                     "Test MAPE (%)": mt["MAPE (%)"],
                     "Hold MAE": mh["MAE"], "Hold RMSE": mh["RMSE"],
                     "Hold MAPE (%)": mh["MAPE (%)"],
                     "Hold Bias (%)": mh["Bias (%)"]})


     # HOLT-WINTERS
    print("\n[INFER] Suav. Exp. HW")

    y_train_val = pd.concat([train, val])["value"].values.astype(float)

    # treino final
    hw_fit = fit_hw(y_train_val, p_hw)

    # TEST walk-forward
    pt_arr = walk_forward_hw(
        y_train_val,
        test["value"].values.astype(float),
        p_hw
    )

    pt_hw = pd.Series(pt_arr, index=test.index)

    # HOLDOUT recursivo
    if len(holdout):

        y_hist = pd.concat([train, val, test])["value"].values.astype(float)

        ph_arr = recursive_hw(
            y_hist,
            len(holdout),
            p_hw
        )

        ph_hw = pd.Series(ph_arr, index=holdout.index)

    else:
        ph_hw = pd.Series(dtype=float)

    all_preds["Suav. Exp. HW"] = {
        "test": pt_hw,
        "holdout": ph_hw
    }

    mt = metrics(test["value"].values, pt_hw.values)

    mh = (
        metrics(holdout["value"].values, ph_hw.values)
        if len(holdout)
        else {k: np.nan for k in ["MAE","RMSE","MAPE (%)","Bias (%)"]}
    )

    print(f"   TEST    MAE={mt['MAE']:.1f}  MAPE={mt['MAPE (%)']:.2f}%")
    print(f"   HOLDOUT MAE={mh['MAE']:.1f}  MAPE={mh['MAPE (%)']:.2f}%")

    rows.append({
        "Modelo": "Suav. Exp. HW",
        "Test MAE": mt["MAE"],
        "Test RMSE": mt["RMSE"],
        "Test MAPE (%)": mt["MAPE (%)"],
        "Hold MAE": mh["MAE"],
        "Hold RMSE": mh["RMSE"],
        "Hold MAPE (%)": mh["MAPE (%)"],
        "Hold Bias (%)": mh["Bias (%)"]
    })


    # ---------- SAÍDAS ----------
    summary = pd.DataFrame(rows).sort_values("Test MAPE (%)")
    summary_path = os.path.join(OUTPUT_DIR, "classicos_metrics.csv")
    summary.to_csv(summary_path, index=False)
    print("\n========================== RESUMO ==========================")
    print(summary.round(3).to_string(index=False))
    print(f"\n[OUT] métricas salvas em {summary_path}")

    # CSV consolidado com todas as previsões
    rows_pred = {"real": df["value"]}
    for name, d in all_preds.items():
        s = pd.concat([d["test"], d["holdout"]]).reindex(df.index)
        rows_pred[name] = s
    pd.DataFrame(rows_pred).to_csv(os.path.join(OUTPUT_DIR, "classicos_previsoes.csv"))
    print(f"[OUT] previsões salvas em {os.path.join(OUTPUT_DIR, 'classicos_previsoes.csv')}")

    plot_comparativo(df, all_preds,
                     os.path.join(OUTPUT_DIR, "classicos_comparativo.png"))

    plot_test_predictions(
    df,
    test,
    all_preds,
    os.path.join(OUTPUT_DIR, "classicos_test_2017_2019.png")
)

    return summary, all_preds

main()

[LOAD] base...
   train=149  val=12  test=36  holdout=72
   X_train=(125, 12)  X_val=(12, 12)

[OPTUNA] otimizando hparams (val = 2016)...
   [LinearRegression] sem hiperparâmetros para otimizar
   [OPTUNA] SVR    MSE val = 0.208678 | params = {'C': 17.54885971324522, 'gamma': 0.040746755790793225, 'epsilon': 0.1274766304194478}
   [OPTUNA] GBRT   MSE val = 0.211920 | params = {'n_estimators': 750, 'max_depth': 5, 'lr': 0.01354635195439923, 'subsample': 0.9931073776669301, 'min_samples_leaf': 2}
   [OPTUNA] LWR    MSE val = 0.220110 | params = {'bandwidth': 4.986352369091248}
   [OPTUNA] KNN    MSE val = 0.323344 | params = {'n_neighbors': 2}
   [OPTUNA] SMA    MSE val = 0.265340 | params = {'k': 11}
   [OPTUNA] HW MSE val = 3024765.208443 | params = {'trend': 'mul', 'seasonal': 'mul', 'seasonal_periods': 24}

[FIT] treinando modelos finais com train+val...

[INFER] Linear Regression
   TEST    MAE=2287.9  MAPE=4.69%
   HOLDOUT MAE=10444.2  MAPE=13.70%

[INFER] SVR
   TEST    MAE=2020.

(              Modelo     Test MAE    Test RMSE  Test MAPE (%)      Hold MAE  \
 6        Media Movel  1496.845455  1832.693534       3.129451  10859.298914   
 5                KNN  1755.416667  2250.491234       3.522689  13035.250694   
 1                SVR  2019.969303  2671.590656       4.168340  13054.096623   
 4     Nearest (1-NN)  2106.305556  2895.145216       4.244055  11358.834722   
 7          AutoARIMA  2147.840951  2660.581670       4.324671  11544.939411   
 3         Lazy (LWR)  2281.503856  2944.473665       4.677072  10507.369347   
 0  Linear Regression  2287.872060  2953.206009       4.686953  10444.204461   
 2               GBRT  2342.209873  3218.513732       4.779414  12181.401625   
 8      Suav. Exp. HW  2724.451155  3390.877277       5.493352  11638.325627   
 
       Hold RMSE  Hold MAPE (%)  Hold Bias (%)  
 6  15443.366510      14.165024       3.959131  
 5  17901.811337      16.753973       8.931716  
 1  18056.098347      16.699162      10.028174  
 4

## Etapa 3 — Visualizações

Lê os CSVs gerados nas Etapas 1 e 2 e produz 5 gráficos comparativos:
- `01_holdout_redes_neurais.png`
- `02_holdout_classicos.png`
- `03_holdout_todos.png`
- `04_metricas_comparativo.png`
- `05_analise_serie.png`

In [25]:
def load_predictions() -> pd.DataFrame:
    print(f"[LOAD] {CLASSICOS_CSV}")
    c = pd.read_csv(CLASSICOS_CSV, index_col=0, parse_dates=True)
    print(f"   colunas no CSV: {list(c.columns)}")

    df = pd.DataFrame(index=c.index)
    df["Real"] = c["real"]
    for m in CLASSIC_MODELS:
        if m in c.columns:
            df[m] = c[m]

    if os.path.exists(NEURAIS_CSV):
        print(f"\n[LOAD] {NEURAIS_CSV}")
        nr = pd.read_csv(NEURAIS_CSV, index_col=0, parse_dates=True)
        print(f"   colunas no CSV: {list(nr.columns)}")

        # alinha índices
        idx = df.index.union(nr.index)
        df = df.reindex(idx)
        if "real" in nr.columns:
            df["Real"] = df["Real"].combine_first(nr["real"].reindex(idx))

        for m in NEURAL_MODELS:
            if m not in nr.columns:
                print(f"   [AUSENTE] '{m}' — coluna NÃO existe no CSV. "
                      f"Re-execute forecast_lstm_tft.py?")
                continue
            serie = nr[m].reindex(idx)
            n_valid = serie.notna().sum()
            if n_valid == 0:
                print(f"   [VAZIO]   '{m}' — coluna existe mas TODOS os "
                      f"valores são NaN (provável falha no treino/predict).")
                continue
            df[m] = serie
            # diagnóstico extra: quantos pontos no test e holdout?
            mask_test = (idx > pd.to_datetime(VAL_END)) & (idx <= pd.to_datetime(TEST_END))
            mask_hold = idx > pd.to_datetime(TEST_END)
            n_test = serie[mask_test].notna().sum()
            n_hold = serie[mask_hold].notna().sum()
            print(f"   [OK]      '{m}' — {n_valid} pts não-nulos "
                  f"(test: {n_test}, holdout: {n_hold})")
    else:
        print(f"\n[AUSENTE] {NEURAIS_CSV} — só os clássicos serão plotados")

    available = [c for c in df.columns if c != "Real"]
    print(f"\n[LOAD] modelos disponíveis para plotar: {available}")
    return df


def compute_metrics(df, models, regime: str) -> pd.DataFrame:
    if regime == "Test":
        mask = (df.index > pd.to_datetime(VAL_END)) & \
               (df.index <= pd.to_datetime(TEST_END))
    elif regime == "Holdout":
        mask = df.index > pd.to_datetime(TEST_END)
    else:
        raise ValueError(regime)

    rows = []
    for m in models:
        if m not in df.columns: continue
        sub = df.loc[mask, ["Real", m]].dropna()
        if len(sub) == 0: continue
        err = sub["Real"] - sub[m]
        rows.append({
            "Modelo": m,
            "MAE":      err.abs().mean(),
            "RMSE":     float(np.sqrt((err ** 2).mean())),
            "MAPE (%)": (err.abs() / sub["Real"] * 100).mean(),
            "Bias (%)": (err / sub["Real"] * 100).mean(),
        })
    return pd.DataFrame(rows)


# =================================================================== #
# Helpers de plot
# =================================================================== #
def _draw_phase_background(ax, span_min, span_max):
    train_end = pd.to_datetime(TRAIN_END) + pd.offsets.MonthEnd(0)
    val_end   = pd.to_datetime(VAL_END)   + pd.offsets.MonthEnd(0)
    test_end  = pd.to_datetime(TEST_END)  + pd.offsets.MonthEnd(0)

    ax.axvspan(span_min, train_end, color="#1f77b4", alpha=0.05)
    ax.axvspan(train_end, val_end,  color="#ff7f0e", alpha=0.08)
    ax.axvspan(val_end,   test_end, color="#2ca02c", alpha=0.08)
    ax.axvspan(test_end,  span_max, color="#d62728", alpha=0.12)


def _annotate_phases(ax, y_frac=0.93):
    train_end = pd.to_datetime(TRAIN_END) + pd.offsets.MonthEnd(0)
    val_end   = pd.to_datetime(VAL_END)   + pd.offsets.MonthEnd(0)
    test_end  = pd.to_datetime(TEST_END)  + pd.offsets.MonthEnd(0)
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    yloc = ylim[0] + (ylim[1] - ylim[0]) * y_frac

    x_min = pd.to_datetime(plt.matplotlib.dates.num2date(xlim[0])).replace(tzinfo=None)
    x_max = pd.to_datetime(plt.matplotlib.dates.num2date(xlim[1])).replace(tzinfo=None)

    for vs, ve, lbl in [
        (x_min,     train_end, "TRAIN"),
        (train_end, val_end,   "VAL"),
        (val_end,   test_end,  "TEST"),
        (test_end,  x_max,     "HOLDOUT"),
    ]:
        mid = vs + (ve - vs) / 2
        ax.text(mid, yloc, lbl, ha="center", va="top", fontsize=8.5,
                color="#444444", fontweight="bold")


# =================================================================== #
# 1. Gráfico principal (contexto + zoom holdout)
# =================================================================== #
def plot_principal(df, models, title, out_path):
    fig = plt.figure(figsize=(14, 8.5))
    gs = fig.add_gridspec(2, 1, height_ratios=[1, 2.2], hspace=0.32)

    # ------ Painel superior: contexto ------
    ax1 = fig.add_subplot(gs[0])
    ax1.plot(df.index, df["Real"], color="black", lw=1.0)
    _draw_phase_background(ax1, df.index.min(), df.index.max())
    _annotate_phases(ax1)
    ax1.set_title("Contexto — Série completa", fontsize=10, loc="left",
                  color="#444444")
    ax1.set_ylabel("Despesa")
    ax1.grid(alpha=0.25)
    ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)

    # ------ Painel inferior: zoom holdout ------
    ax2 = fig.add_subplot(gs[1])
    test_end = pd.to_datetime(TEST_END) + pd.offsets.MonthEnd(0)
    df_h = df[df.index > test_end]

    # MAPE por modelo (para ordenar)
    mapes = {}
    for m in models:
        if m not in df.columns:
            print(f"   [plot] '{m}' não está em df.columns — pulado")
            continue
        sub = df_h[["Real", m]].dropna()
        if len(sub) == 0:
            print(f"   [plot] '{m}' sem dados válidos no holdout — pulado")
            continue
        mapes[m] = np.mean(np.abs((sub["Real"] - sub[m]) / sub["Real"])) * 100
    sorted_models = sorted(mapes.keys(), key=lambda x: mapes[x])

    # Realizado
    ax2.plot(df_h.index, df_h["Real"], color="black", lw=2.3,
             label="Realizado", zorder=20)

    # Predições (melhor mais opaco e mais grosso)
    for i, m in enumerate(sorted_models):
        is_best = (i == 0)
        ax2.plot(df_h.index, df_h[m],
                 color=COLORS.get(m, plt.cm.tab10(i % 10)),
                 lw=2.0 if is_best else 1.3,
                 alpha=0.95 if is_best else 0.65,
                 linestyle="-" if is_best else "--",
                 label=f"{m}  (MAPE {mapes[m]:.2f}%)",
                 zorder=19 - i)

    # Sombreamento sutil do holdout
    ax2.axvspan(df_h.index.min(), df_h.index.max(),
                color="#d62728", alpha=0.04)

    ax2.set_title("Zoom no Holdout (>2019) — Previsão vs Realizado",
                  fontsize=11, fontweight="bold", loc="left")
    ax2.set_xlabel("Data"); ax2.set_ylabel("Despesa")
    ncol = 2 if len(sorted_models) >= 5 else 1
    ax2.legend(loc="upper left", fontsize=9, framealpha=0.95, ncol=ncol)
    ax2.grid(alpha=0.3)
    ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)

    plt.suptitle(title, fontsize=14, fontweight="bold", y=1.00)
    plt.tight_layout()
    plt.savefig(out_path, dpi=130, bbox_inches="tight")
    plt.close()
    print(f"  [PLOT] {out_path}")


# =================================================================== #
# 2. Dashboard de métricas (2×2)
# =================================================================== #
def plot_metrics_dashboard(df, out_path):
    fig, axes = plt.subplots(2, 2, figsize=(15.5, 10.5))

    all_models = [m for m in (NEURAL_MODELS + CLASSIC_MODELS) if m in df.columns]
    m_test = compute_metrics(df, all_models, "Test").set_index("Modelo")
    m_hold = compute_metrics(df, all_models, "Holdout").set_index("Modelo")
    order = m_hold.sort_values("MAPE (%)").index.tolist()

    # ---- (0,0) MAPE Test vs Holdout
    ax = axes[0, 0]
    x = np.arange(len(order)); w = 0.38
    test_vals = m_test.reindex(order)["MAPE (%)"].values
    hold_vals = m_hold.reindex(order)["MAPE (%)"].values
    ax.bar(x - w/2, test_vals, w, color="#1f77b4",
           edgecolor="white", label="Test (2017-2019)")
    ax.bar(x + w/2, hold_vals, w, color="#d62728",
           edgecolor="white", label="Holdout (>2019)")
    for i in range(len(order)):
        ax.text(i - w/2, test_vals[i] + 0.15, f"{test_vals[i]:.1f}",
                ha="center", fontsize=7, color="#1f77b4", fontweight="bold")
        ax.text(i + w/2, hold_vals[i] + 0.15, f"{hold_vals[i]:.1f}",
                ha="center", fontsize=7, color="#d62728", fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=30, ha="right", fontsize=9)
    ax.set_ylabel("MAPE (%)")
    ax.set_title("MAPE por modelo — Test vs Holdout",
                 fontweight="bold", loc="left")
    ax.legend(framealpha=0.95)
    ax.grid(alpha=0.3, axis="y")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    # ---- (0,1) Bias % no Holdout
    ax = axes[0, 1]
    bias_h = m_hold["Bias (%)"].reindex(order).sort_values()
    colors_b = ["#2ca02c" if abs(v) < 3 else
                "#ff7f0e" if abs(v) < 6 else "#d62728"
                for v in bias_h.values]
    ax.barh(np.arange(len(bias_h)), bias_h.values,
            color=colors_b, edgecolor="white")
    ax.set_yticks(np.arange(len(bias_h)))
    ax.set_yticklabels(bias_h.index, fontsize=9)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("Bias (%) — (real - previsto) / real")
    ax.set_title("Bias no Holdout — calibração dos modelos\n"
                 "verde |bias|<3% · laranja <6% · vermelho ≥6%",
                 fontweight="bold", loc="left", fontsize=10)
    ax.grid(alpha=0.3, axis="x")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    max_abs = max(abs(bias_h.min()), abs(bias_h.max())) + 1
    ax.set_xlim(-max_abs, max_abs)
    for i, v in enumerate(bias_h.values):
        ax.text(v + (0.15 * np.sign(v) if v != 0 else 0.15), i,
                f"{v:+.2f}%", va="center",
                ha="left" if v >= 0 else "right", fontsize=8)

    # ---- (1,0) MAPE por ano no Holdout (heatmap)
    ax = axes[1, 0]
    df_h_only = df[df.index > pd.to_datetime(TEST_END)]
    rows = {}
    for m in all_models:
        if m not in df.columns: continue
        sub = df_h_only[["Real", m]].dropna().copy()
        if len(sub) == 0: continue
        sub["year"] = sub.index.year
        rows[m] = sub.groupby("year").apply(
            lambda g: np.mean(np.abs((g["Real"] - g[m]) / g["Real"])) * 100
        )
    heat = pd.DataFrame(rows).T.reindex(order)
    im = ax.imshow(heat.values, aspect="auto", cmap="YlOrRd", vmin=0)
    ax.set_xticks(np.arange(len(heat.columns)))
    ax.set_xticklabels(heat.columns)
    ax.set_yticks(np.arange(len(heat.index)))
    ax.set_yticklabels(heat.index, fontsize=9)
    ax.set_title("MAPE por ano no Holdout (%)",
                 fontweight="bold", loc="left")
    vmax_h = np.nanmax(heat.values)
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            v = heat.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.1f}", ha="center", va="center",
                        fontsize=8.5, fontweight="bold",
                        color="white" if v > vmax_h * 0.55 else "black")
    plt.colorbar(im, ax=ax, label="MAPE (%)", fraction=0.04, pad=0.02)

    # ---- (1,1) Boxplot erros % no Holdout
    ax = axes[1, 1]
    box_data, labels = [], []
    for m in order:
        if m not in df.columns: continue
        sub = df_h_only[["Real", m]].dropna()
        if len(sub) == 0: continue
        e = ((sub["Real"] - sub[m]) / sub["Real"] * 100).values
        box_data.append(e); labels.append(m)
    bp = ax.boxplot(box_data, labels=labels, patch_artist=True, vert=False,
                    medianprops=dict(color="black", linewidth=1.4),
                    boxprops=dict(linewidth=1.0),
                    whiskerprops=dict(linewidth=1.0))
    for patch, m in zip(bp["boxes"], labels):
        patch.set_facecolor(COLORS.get(m, "#888888"))
        patch.set_alpha(0.65)
    ax.axvline(0, color="black", lw=1.0, linestyle="--", alpha=0.7)
    ax.set_xlabel("Erro percentual (%)  —  (real - previsto)/real")
    ax.set_title("Distribuição dos erros no Holdout",
                 fontweight="bold", loc="left")
    ax.grid(alpha=0.3, axis="x")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    plt.suptitle("Análise Comparativa — Métricas",
                 fontsize=14, fontweight="bold", y=1.005)
    plt.tight_layout()
    plt.savefig(out_path, dpi=130, bbox_inches="tight")
    plt.close()
    print(f"  [PLOT] {out_path}")


# =================================================================== #
# 3. Análise da série (STL + sazonalidade + YoY)
# =================================================================== #
def plot_series_analysis(df, out_path):
    real = df["Real"].dropna()

    fig = plt.figure(figsize=(15, 13))
    if HAS_STL:
        gs = fig.add_gridspec(5, 2, height_ratios=[1, 1, 1, 1, 1.3],
                              hspace=0.50, wspace=0.22)
        stl = STL(real, period=12, robust=True).fit()

        ax_a = fig.add_subplot(gs[0, :])
        ax_b = fig.add_subplot(gs[1, :], sharex=ax_a)
        ax_c = fig.add_subplot(gs[2, :], sharex=ax_a)
        ax_d = fig.add_subplot(gs[3, :], sharex=ax_a)

        ax_a.plot(real.index, real.values, color="black", lw=1.1)
        ax_a.set_title("Série Original", loc="left", fontweight="bold")
        ax_a.set_ylabel("Despesa")

        ax_b.plot(stl.trend.index, stl.trend.values, color="#1f77b4", lw=1.4)
        ax_b.set_title("Tendência (STL)", loc="left", fontweight="bold")
        ax_b.set_ylabel("Tendência")

        ax_c.plot(stl.seasonal.index, stl.seasonal.values, color="#2ca02c", lw=0.9)
        ax_c.axhline(0, color="black", lw=0.5, alpha=0.6)
        ax_c.set_title("Componente Sazonal (STL)", loc="left", fontweight="bold")
        ax_c.set_ylabel("Sazonal")

        ax_d.plot(stl.resid.index, stl.resid.values, color="#d62728", lw=0.7)
        ax_d.axhline(0, color="black", lw=0.5, alpha=0.6)
        ax_d.set_title("Resíduo (STL) — o que sobra após retirar tendência e sazonalidade",
                       loc="left", fontweight="bold")
        ax_d.set_ylabel("Resíduo"); ax_d.set_xlabel("Data")

        for axx in [ax_a, ax_b, ax_c, ax_d]:
            axx.grid(alpha=0.3)
            axx.spines["top"].set_visible(False)
            axx.spines["right"].set_visible(False)
            for v in [TRAIN_END, VAL_END, TEST_END]:
                axx.axvline(pd.to_datetime(v) + pd.offsets.MonthEnd(0),
                            color="gray", linestyle=":", alpha=0.5)

        ax_month = fig.add_subplot(gs[4, 0])
        ax_yoy   = fig.add_subplot(gs[4, 1])
    else:
        gs = fig.add_gridspec(2, 2, height_ratios=[1.5, 1], hspace=0.35)
        ax_top = fig.add_subplot(gs[0, :])
        ax_top.plot(real.index, real.values, color="black", lw=1.1)
        ax_top.set_title("Série Original (statsmodels indisponível para STL)",
                         loc="left", fontweight="bold")
        ax_top.grid(alpha=0.3)
        ax_month = fig.add_subplot(gs[1, 0])
        ax_yoy   = fig.add_subplot(gs[1, 1])

    # Sazonalidade mensal (boxplot por mês)
    month_groups = [real[real.index.month == m].values for m in range(1, 13)]
    bp = ax_month.boxplot(month_groups, labels=list(range(1, 13)),
                          patch_artist=True,
                          medianprops=dict(color="black", linewidth=1.2))
    for patch in bp["boxes"]:
        patch.set_facecolor("#1f77b4"); patch.set_alpha(0.55)
    ax_month.set_title("Sazonalidade — distribuição mensal (todos os anos)",
                       loc="left", fontweight="bold")
    ax_month.set_xlabel("Mês"); ax_month.set_ylabel("Despesa")
    ax_month.grid(alpha=0.3, axis="y")
    ax_month.spines["top"].set_visible(False)
    ax_month.spines["right"].set_visible(False)

    # Crescimento ano-a-ano
    yearly = real.resample("Y").sum()
    yoy = (yearly.pct_change() * 100).dropna()
    yoy = yoy[yoy.index.year != 2004]
    bar_colors = ["#2ca02c" if v > 0 else "#d62728" for v in yoy.values]
    ax_yoy.bar(yoy.index.year, yoy.values, color=bar_colors,
               edgecolor="white")
    ax_yoy.axhline(0, color="black", lw=0.8)
    ax_yoy.set_title("Crescimento ano-a-ano da despesa anual (%)",
                     loc="left", fontweight="bold")
    ax_yoy.set_xlabel("Ano"); ax_yoy.set_ylabel("Crescimento (%)")
    ax_yoy.grid(alpha=0.3, axis="y")
    ax_yoy.spines["top"].set_visible(False)
    ax_yoy.spines["right"].set_visible(False)
    for x, v in zip(yoy.index.year, yoy.values):
        ax_yoy.text(x, v + (0.4 if v >= 0 else -1.2), f"{v:.1f}%",
                    ha="center", fontsize=7,
                    color="#2ca02c" if v > 0 else "#d62728",
                    fontweight="bold")

    plt.suptitle("Análise da Série de Despesas — Componentes e Sazonalidade",
                 fontsize=14, fontweight="bold", y=1.005)
    plt.tight_layout()
    plt.savefig(out_path, dpi=130, bbox_inches="tight")
    plt.close()
    print(f"  [PLOT] {out_path}")


# =================================================================== #
# Main
# =================================================================== #
def main():
    print("[CHARTS] Carregando previsões...\n")
    df = load_predictions()
    print(f"\n  Período: {df.index.min().date()} -> {df.index.max().date()}")

    # Filtra os modelos efetivamente disponíveis
    neural_available  = [m for m in NEURAL_MODELS  if m in df.columns]
    classic_available = [m for m in CLASSIC_MODELS if m in df.columns]
    print(f"  Neurais:   {neural_available}")
    print(f"  Clássicos: {classic_available}")

    if not neural_available:
        print("\n[!] NENHUM modelo neural disponível — gráfico 01 ficará vazio.")
        print("    Verifique se forecast_lstm_tft.py foi executado com sucesso")
        print("    e se neurais_previsoes.csv contém as colunas esperadas.\n")

    print("\n[CHARTS] Gerando 3 gráficos principais (holdout)...")
    plot_principal(
        df, neural_available,
        "Holdout — Redes Neurais (LSTM isolado vs TFT)",
        os.path.join(CHARTS_DIR, "01_holdout_redes_neurais.png"),
    )
    plot_principal(
        df, classic_available,
        "Holdout — Modelos Clássicos",
        os.path.join(CHARTS_DIR, "02_holdout_classicos.png"),
    )
    plot_principal(
        df, neural_available + classic_available,
        "Holdout — Comparação completa (Redes Neurais + Clássicos)",
        os.path.join(CHARTS_DIR, "03_holdout_todos.png"),
    )

    print("\n[CHARTS] Dashboard de métricas...")
    plot_metrics_dashboard(df, os.path.join(CHARTS_DIR, "04_metricas_comparativo.png"))

    print("\n[CHARTS] Análise da série...")
    plot_series_analysis(df, os.path.join(CHARTS_DIR, "05_analise_serie.png"))

    print(f"\n[CHARTS] Pronto. Arquivos em {CHARTS_DIR}/")

main()

[CHARTS] Carregando previsões...

[LOAD] ./outputs/classicos_previsoes.csv
   colunas no CSV: ['real', 'Linear Regression', 'SVR', 'GBRT', 'Lazy (LWR)', 'Nearest (1-NN)', 'KNN', 'Media Movel', 'AutoARIMA', 'Suav. Exp. HW']

[LOAD] ./outputs/neurais_previsoes.csv
   colunas no CSV: ['real', 'LSTM isolado']
   [OK]      'LSTM isolado' — 108 pts não-nulos (test: 36, holdout: 72)
   [AUSENTE] 'TFT' — coluna NÃO existe no CSV. Re-execute forecast_lstm_tft.py?

[LOAD] modelos disponíveis para plotar: ['AutoARIMA', 'Linear Regression', 'SVR', 'GBRT', 'Lazy (LWR)', 'Nearest (1-NN)', 'KNN', 'Media Movel', 'Suav. Exp. HW', 'LSTM isolado']

  Período: 2003-08-01 -> 2025-12-01
  Neurais:   ['LSTM isolado']
  Clássicos: ['AutoARIMA', 'Linear Regression', 'SVR', 'GBRT', 'Lazy (LWR)', 'Nearest (1-NN)', 'KNN', 'Media Movel', 'Suav. Exp. HW']

[CHARTS] Gerando 3 gráficos principais (holdout)...
  [PLOT] ./outputs/charts/01_holdout_redes_neurais.png
  [PLOT] ./outputs/charts/02_holdout_classicos.png
  [